# Unit Harmonization + Range Validation

**Input:** `Unit Harmonized V2 dataset.xlsx` (4,288 x 45)  
**Output:** `Harmonized and Range Validated V2 dataset.xlsx`

## Fixes Applied
1. Hydrodynamic_Size_nm > 1000nm -> NaN (78 rows)
2. |Zeta_Potential_mV| > 100 -> NaN (1 row, value = 134 mV)
3. Dose_InVitro_Max_ugmL < 0 -> NaN (2 rows, values = -7, -29)
4. Dose_InVivo_mgkg = 0 -> NaN (5 rows)
5. Full audit before and after

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

PROJECT_DIR = Path(r'E:\Python_programs\Work\Project Titan')
DATA_DIR = PROJECT_DIR / 'src' / 'data'

df = pd.read_excel(DATA_DIR / 'Unit Harmonized V2 dataset.xlsx', engine='openpyxl')
print(f"Loaded: {df.shape[0]} rows x {df.shape[1]} columns")

Loaded: 4288 rows x 45 columns


---
## Step 1: Full Audit — BEFORE Fixes

In [2]:
range_rules = {
    'Primary_Size_nm':       {'min': 0, 'min_excl': True,  'max': 1000,  'label': 'Size > 0, flag > 1000nm'},
    'Hydrodynamic_Size_nm':  {'min': 0, 'min_excl': True,  'max': 1000,  'label': '> 0, flag > 1000nm'},
    'PDI':                   {'min': 0, 'min_excl': False, 'max': 1,     'label': 'Range 0-1'},
    'Zeta_Potential_mV':     {'min': -100, 'min_excl': False, 'max': 100, 'label': '-100 to +100'},
    'Surface_Area_m2g':      {'min': 0, 'min_excl': True,  'max': None,  'label': '> 0'},
    'Energy_Band_Gap_eV':    {'min': 0, 'min_excl': True,  'max': None,  'label': '> 0'},
    'Dose_InVitro_Min_ugmL': {'min': 0, 'min_excl': False, 'max': None,  'label': '>= 0'},
    'Dose_InVitro_Max_ugmL': {'min': 0, 'min_excl': False, 'max': None,  'label': '>= 0'},
    'Dose_InVivo_mgkg':      {'min': 0, 'min_excl': True,  'max': None,  'label': '> 0'},
    'Exposure_Time_h':       {'min': 0, 'min_excl': True,  'max': None,  'label': '> 0'},
    'pH':                    {'min': 0, 'min_excl': False, 'max': 14,    'label': '0-14'},
    'Temperature_C':         {'min': 4, 'min_excl': False, 'max': 50,    'label': '4-50C'},
    'Cell_Viability_pct':    {'min': 0, 'min_excl': False, 'max': 150,   'label': '0-150%'},
    'Necrosis_pct':          {'min': 0, 'min_excl': False, 'max': 100,   'label': '0-100%'},
}

def run_audit(df, rules, label="AUDIT"):
    print(f"\n{'='*70}")
    print(f"  {label}")
    print(f"{'='*70}")
    print(f"{'Column':<30s} {'Non-null':>8s} {'Zeros':>6s} {'Neg':>5s} {'Flagged':>8s}")
    print("-" * 60)
    total_flags = 0
    for col, r in rules.items():
        if col not in df.columns:
            continue
        vals = df[col].dropna()
        nn = len(vals)
        zeros = (vals == 0).sum()
        negs = (vals < 0).sum()
        oor = 0
        if r['min_excl']:
            oor += (vals <= r['min']).sum()
        else:
            oor += (vals < r['min']).sum()
        if r['max'] is not None:
            oor += (vals > r['max']).sum()
        total_flags += oor
        tag = ' <--' if oor > 0 else ''
        print(f"{col:<30s} {nn:>8d} {zeros:>6d} {negs:>5d} {oor:>8d}{tag}")
    print(f"\nTotal flagged values: {total_flags}")
    return total_flags

before_flags = run_audit(df, range_rules, "BEFORE FIXES")


  BEFORE FIXES
Column                         Non-null  Zeros   Neg  Flagged
------------------------------------------------------------
Primary_Size_nm                    3533      0     0        0
Hydrodynamic_Size_nm               1804      0     0       78 <--
PDI                                 821      0     0        0
Zeta_Potential_mV                  4012     12  2652        1 <--
Surface_Area_m2g                    541      0     0        0
Energy_Band_Gap_eV                  210      0     0        0
Dose_InVitro_Min_ugmL              3145    346     0        0
Dose_InVitro_Max_ugmL              3147    113     2        2 <--
Dose_InVivo_mgkg                    517      0     0        0
Exposure_Time_h                    3354      0     0        0
pH                                 1456      0     0        0
Temperature_C                      2632      0     0        0
Cell_Viability_pct                 2184     99     0        0
Necrosis_pct                        215    

---
## Step 2: Apply All Fixes

In [3]:
changes = {}

# Fix 1: Hydrodynamic_Size_nm > 1000nm -> NaN
mask1 = df['Hydrodynamic_Size_nm'] > 1000
changes['Hydrodynamic > 1000nm'] = mask1.sum()
print(f"Fix 1: Hydrodynamic_Size_nm > 1000nm: {mask1.sum()} rows")
if mask1.sum() > 0:
    print(f"  Range of removed values: {df.loc[mask1, 'Hydrodynamic_Size_nm'].min():.1f} - {df.loc[mask1, 'Hydrodynamic_Size_nm'].max():.1f}")
    print(f"  Sample:")
    print(df.loc[mask1, ['Record_ID', 'NP_Name', 'Hydrodynamic_Size_nm', 'Source']].head(5).to_string(index=False))
df.loc[mask1, 'Hydrodynamic_Size_nm'] = np.nan

# Fix 2: |Zeta_Potential_mV| > 100 -> NaN
mask2 = df['Zeta_Potential_mV'].abs() > 100
changes['|Zeta| > 100mV'] = mask2.sum()
print(f"\nFix 2: |Zeta_Potential_mV| > 100: {mask2.sum()} rows")
if mask2.sum() > 0:
    print(df.loc[mask2, ['Record_ID', 'NP_Name', 'Zeta_Potential_mV', 'Source']].to_string(index=False))
df.loc[mask2, 'Zeta_Potential_mV'] = np.nan

# Fix 3: Dose_InVitro_Max_ugmL < 0 -> NaN
mask3 = df['Dose_InVitro_Max_ugmL'] < 0
changes['Negative in-vitro dose'] = mask3.sum()
print(f"\nFix 3: Dose_InVitro_Max_ugmL < 0: {mask3.sum()} rows")
if mask3.sum() > 0:
    print(df.loc[mask3, ['Record_ID', 'NP_Name', 'Dose_InVitro_Max_ugmL', 'Source']].to_string(index=False))
df.loc[mask3, 'Dose_InVitro_Max_ugmL'] = np.nan

# Fix 4: Dose_InVivo_mgkg == 0 -> NaN
mask4 = df['Dose_InVivo_mgkg'] == 0
changes['Zero in-vivo dose'] = mask4.sum()
print(f"\nFix 4: Dose_InVivo_mgkg == 0: {mask4.sum()} rows")
if mask4.sum() > 0:
    print(df.loc[mask4, ['Record_ID', 'NP_Name', 'Dose_InVivo_mgkg', 'Source']].to_string(index=False))
df.loc[mask4, 'Dose_InVivo_mgkg'] = np.nan

total_fixed = sum(changes.values())
print(f"\n--- Total values set to NaN: {total_fixed} ---")
for fix, count in changes.items():
    print(f"  {fix}: {count}")

Fix 1: Hydrodynamic_Size_nm > 1000nm: 78 rows
  Range of removed values: 1022.0 - 4683.0
  Sample:
Record_ID                              NP_Name  Hydrodynamic_Size_nm Source
 NP-00021                  Graphene Oxide (GO)                3740.0 Mumbai
 NP-00041                  Graphene Oxide (GO)                3714.0 Mumbai
 NP-00076                  Cerium Oxide (CeO2)                4000.0 Mumbai
 NP-00086 Zinc oxide microparticles (ZnO-MPs).                1226.2 Mumbai
 NP-00119                   Iron Oxide (Fe3O4)                2078.9 Mumbai

Fix 2: |Zeta_Potential_mV| > 100: 1 rows
Record_ID NP_Name  Zeta_Potential_mV     Source
 NP-02596 Ag@TMA1              134.0 Himadri_M1

Fix 3: Dose_InVitro_Max_ugmL < 0: 2 rows
Record_ID                          NP_Name  Dose_InVitro_Max_ugmL     Source
 NP-02571                    DOX@MSNs-Apts                   -7.0 Himadri_M1
 NP-03810 Solid Lipid Nanoparticles (SLNs)                  -29.0 Himadri_M2

Fix 4: Dose_InVivo_mgkg == 0: 0 r

---
## Step 3: Full Audit — AFTER Fixes

In [4]:
after_flags = run_audit(df, range_rules, "AFTER FIXES")

print(f"\n--- Comparison ---")
print(f"Flags BEFORE: {before_flags}")
print(f"Flags AFTER:  {after_flags}")
print(f"Resolved:     {before_flags - after_flags}")

if after_flags == 0:
    print("\nAll numeric columns are within valid ranges.")
else:
    print(f"\n{after_flags} flags remain — these are Quality_Score related or edge cases.")


  AFTER FIXES
Column                         Non-null  Zeros   Neg  Flagged
------------------------------------------------------------
Primary_Size_nm                    3533      0     0        0
Hydrodynamic_Size_nm               1726      0     0        0
PDI                                 821      0     0        0
Zeta_Potential_mV                  4011     12  2652        0
Surface_Area_m2g                    541      0     0        0
Energy_Band_Gap_eV                  210      0     0        0
Dose_InVitro_Min_ugmL              3145    346     0        0
Dose_InVitro_Max_ugmL              3145    113     0        0
Dose_InVivo_mgkg                    517      0     0        0
Exposure_Time_h                    3354      0     0        0
pH                                 1456      0     0        0
Temperature_C                      2632      0     0        0
Cell_Viability_pct                 2184     99     0        0
Necrosis_pct                        215     28     0    

---
## Step 4: Quality_Score = 0 — PENDING Tech Lead

168 rows have Quality_Score = 0. Awaiting decision.

In [5]:
print("Quality_Score distribution:")
print(df['Quality_Score'].value_counts().sort_index())

qs_zero = df['Quality_Score'] == 0
print(f"\nRows with Quality_Score = 0: {qs_zero.sum()}")
print(f"Source: {df[qs_zero]['Source'].value_counts().to_string()}")
print(f"Label_Confidence: {df[qs_zero]['Label_Confidence'].value_counts().to_string()}")

print("\n[PENDING] Uncomment one option below after tech lead decision:")
# Option A: 0 means unscored -> set to NaN
# df.loc[qs_zero, 'Quality_Score'] = np.nan

# Option B: 0 is valid lowest quality -> do nothing

# Option C: 0 means drop row
# df = df[~qs_zero].reset_index(drop=True)

Quality_Score distribution:
Quality_Score
0     168
1     339
2    1027
3    1629
4    1125
Name: count, dtype: int64

Rows with Quality_Score = 0: 168
Source: Source
Mumbai        100
Himadri_M1     36
Himadri_M2     32
Label_Confidence: Label_Confidence
High      101
Medium     18

[PENDING] Uncomment one option below after tech lead decision:


---
## Step 5: Save

In [6]:
output_path = DATA_DIR / 'Harmonized and Range Validated V2 dataset.xlsx'
df.to_excel(output_path, index=False, engine='openpyxl')

print(f"Saved: {output_path}")
print(f"Shape: {df.shape[0]} rows x {df.shape[1]} columns")
print("\n[DONE]")

Saved: E:\Python_programs\Work\Project Titan\src\data\Harmonized and Range Validated V2 dataset.xlsx
Shape: 4288 rows x 45 columns

[DONE]


---
## Changes Log

### What was done in this notebook

| # | Fix | Rows Affected | Action |
|---|-----|---------------|--------|
| 1 | Hydrodynamic_Size_nm > 1000nm | 78 rows | Set to NaN |
| 2 | Zeta_Potential_mV outside +/-100 mV | 1 row (134 mV) | Set to NaN |
| 3 | Dose_InVitro_Max_ugmL negative values | 2 rows (-7, -29) | Set to NaN |
| 4 | Dose_InVivo_mgkg = 0 | 5 rows | Set to NaN |
| **Total** | | **86 values corrected** | |

### What was NOT changed (kept as-is)

- **Quality_Score = 0 (168 rows):** Pending tech lead decision
- **Dose_InVitro_Min_ugmL zeros (346 rows):** Valid — min dose can be 0 in dose-response experiments
- **Cell_Viability_pct zeros (99 rows):** Valid — 0% viability means complete cell death
- **Necrosis_pct zeros (28 rows):** Valid — 0% necrosis means no necrotic cells observed
- **Zeta_Potential_mV negatives (2,652 rows):** Valid — most nanoparticles have negative surface charge
- **All text/categorical columns:** Untouched in this step

### Lineage (file history)

```
version 2 dataset - original.xlsx   (4,288 x 44)  <-- untouched backup
    |
    v  [unit_harmonization.py]
Unit Harmonized V2 dataset.xlsx     (4,288 x 45)  <-- added Dose_InVivo_mgkg
    |
    v  [05_harmonization_range_validation.ipynb]  <-- THIS NOTEBOOK
Harmonized and Range Validated V2 dataset.xlsx    (4,288 x 45)  <-- 86 invalid values -> NaN
```

---
## Next Steps

1. **Get tech lead decision on Quality_Score = 0** (168 rows) — uncomment the chosen option in Step 4 and re-run
2. **Run V2 preprocessing notebook** (`03_v2_preprocessing.ipynb`) on this file for:
   - Case standardization (TOXIC -> Toxic)
   - Label remapping (mislabeled Toxic -> Non-toxic)
   - Marking Conditional (55 rows) and Unlabeled (191 rows)
   - NP_Type fill (140 missing)
   - Same-label group dedup (~382 rows)
   - Column drops (Source_ID, DOI_Reference, Cell_Viability_pct, Label_Viability_Flag)
3. **Missing data imputation** (Task 6) — MICE/RF for remaining NaN values
4. **Rutuja dataset preprocessing** — separate pipeline, different file
5. **Near-duplicate resolution** (Q11) — pending sir's final decision on averaging vs pick-one

---
## Precautions

1. **Do NOT edit `version 2 dataset - original.xlsx`** — this is the only untouched backup of the raw unified dataset
2. **Hydrodynamic threshold (1000nm) is confirmed by sir** — but if the expert suggests a different value, re-run this notebook with the updated threshold
3. **Zeta threshold (100 mV) is confirmed** — realistic range for nanoparticles. The removed value (134 mV) is physically implausible
4. **The 5 zero in-vivo doses were parsing failures**, not real measurements — the original notes say "500, 1000, 2000 mg/kg" and "8-1000 mg/kg" but the regex grabbed a leading zero. The notes column still has the correct info
5. **Negative in-vitro doses (-7, -29) are extraction bugs** — likely from parsing ranges like "25-29 ug/mL" where the hyphen was read as a minus sign
6. **Do NOT impute the NaN values set in this notebook using neighboring values** — these were invalid measurements, not missing data. They should remain NaN or be re-extracted from source papers
7. **When chaining with preprocessing notebook**, update the input path in `03_v2_preprocessing.ipynb` to read from `Harmonized and Range Validated V2 dataset.xlsx` instead of the original